# 🎓 Student Performance Prediction (Regression + Classification)
**Regression:** `average_score` | **Classification:** `pass_fail`

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              accuracy_score, classification_report, roc_auc_score, roc_curve)
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 2. Load Dataset

In [ ]:
try:
    df = pd.read_csv('students.csv')
except FileNotFoundError:
    print("CSV not found."); raise
print(f"Shape: {df.shape}"); df.head()

## 3. Identify Data Types

In [ ]:
print("Data types:"); print(df.dtypes)
print(f"\nNumeric : {df.select_dtypes(include='number').columns.tolist()}")
print(f"Object  : {df.select_dtypes(include='object').columns.tolist()}")

## 4. Descriptive Statistics

In [ ]:
df.describe().round(2)

## 5. Handle Missing Values

In [ ]:
print("Missing:"); print(df.isnull().sum())
for col in df.columns:
    if df[col].isnull().sum()>0:
        if df[col].dtype=='object': df[col].fillna(df[col].mode()[0],inplace=True); print(f"  '{col}' → mode")
        else: df[col].fillna(df[col].median(),inplace=True); print(f"  '{col}' → median")
print(f"Remaining: {df.isnull().sum().sum()}")

## 6. Handle Duplicates

In [ ]:
print(f"Duplicates: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True); print(f"Shape: {df.shape}")

## 7. Outlier Detection & Handling

In [ ]:
score_cols=['math score','reading score','writing score']
fig,axes=plt.subplots(1,3,figsize=(12,4))
for ax,col in zip(axes,score_cols):
    ax.boxplot(df[col],vert=False,patch_artist=True,boxprops=dict(facecolor='steelblue',alpha=0.6))
    ax.set_title(col,fontsize=10); ax.set_yticks([])
plt.suptitle('Score Boxplots',fontsize=12,y=1.02); plt.tight_layout(); plt.show()
before=len(df)
for col in score_cols:
    Q1,Q3=df[col].quantile([0.25,0.75]); IQR=Q3-Q1
    df=df[df[col].between(Q1-1.5*IQR,Q3+1.5*IQR)]
print(f"Removed: {before-len(df)} | Shape: {df.shape}")

## 8. Feature Engineering — Create Targets

In [ ]:
PASS_THRESHOLD=60
df['average_score']=(df['math score']+df['reading score']+df['writing score'])/3
df['average_score']=df['average_score'].round(2)
df['pass_fail']=(df['average_score']>=PASS_THRESHOLD).astype(int)
print(f"Avg score mean: {df['average_score'].mean():.2f} | Pass rate: {df['pass_fail'].mean()*100:.1f}%")

## 9. Visualizations & Insights

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4))
axes[0].hist(df['average_score'],bins=25,color='steelblue',edgecolor='white')
axes[0].axvline(PASS_THRESHOLD,color='red',linestyle='--',linewidth=2,label=f'Pass ({PASS_THRESHOLD})')
axes[0].axvline(df['average_score'].mean(),color='orange',linestyle='--',label=f"Mean: {df['average_score'].mean():.1f}")
axes[0].set_title('Average Score Distribution'); axes[0].legend(fontsize=8)

df.boxplot(column='math score',by='test preparation course',ax=axes[1])
plt.sca(axes[1]); plt.title('Math Score by Test Prep')

edu_order=['some high school','high school','some college',"associate's degree","bachelor's degree","master's degree"]
edu_means=df.groupby('parental level of education')['average_score'].mean().reindex(edu_order)
axes[2].barh(edu_means.index,edu_means.values,color='#2ecc71',edgecolor='white')
axes[2].set_title("Score by Parental Education"); axes[2].set_yticklabels(edu_order,fontsize=7)

plt.suptitle(''); plt.tight_layout(); plt.show()
print("""Insights:
1. Most students score 60–85; very few below pass threshold.
2. Students who completed test prep score noticeably higher.
3. Higher parental education level correlates with higher student scores.""")

## 10. Encode Categorical & Scale Features

In [ ]:
df_enc=df.copy()
cat_cols=df_enc.select_dtypes(include='object').columns.tolist()
print(f"Encoding: {cat_cols}")
df_enc=pd.get_dummies(df_enc,columns=cat_cols,drop_first=True)

drop_cols=['math score','reading score','writing score','average_score','pass_fail']
X=df_enc.drop(columns=drop_cols)
y_reg=df_enc['average_score']; y_clf=df_enc['pass_fail']
FEATURE_COLS=X.columns.tolist()

X_train,X_test,yr_train,yr_test,yc_train,yc_test=train_test_split(
    X,y_reg,y_clf,test_size=0.2,random_state=42,stratify=y_clf)
scaler=StandardScaler()
X_train_sc=scaler.fit_transform(X_train); X_test_sc=scaler.transform(X_test)
print(f"Features: {FEATURE_COLS}")
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 11. Regression Models

In [ ]:
reg_models={
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Random Forest'     : RandomForestRegressor(n_estimators=100,max_depth=6,random_state=42,n_jobs=-1),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=100,learning_rate=0.1,max_depth=4,random_state=42)
}
reg_results={}
for name,model in reg_models.items():
    model.fit(X_train_sc,yr_train); yp=model.predict(X_test_sc)
    reg_results[name]={'RMSE':round(np.sqrt(mean_squared_error(yr_test,yp)),3),
                       'MAE' :round(mean_absolute_error(yr_test,yp),3),
                       'R²'  :round(r2_score(yr_test,yp),4)}
    print(f'✓ {name}')

## 12. Classification Models

In [ ]:
clf_models={
    'Logistic Regression': LogisticRegression(max_iter=1000,random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=5,random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100,max_depth=6,random_state=42)
}
clf_results={}; clf_preds={}; clf_probs={}
for name,model in clf_models.items():
    model.fit(X_train_sc,yc_train)
    yp=model.predict(X_test_sc); ypr=model.predict_proba(X_test_sc)[:,1]
    clf_preds[name]=yp; clf_probs[name]=ypr
    clf_results[name]={'Accuracy':round(accuracy_score(yc_test,yp),4),'ROC-AUC':round(roc_auc_score(yc_test,ypr),4)}
    print(f"\n{'='*40}\n  {name}\n{'='*40}")
    print(classification_report(yc_test,yp,target_names=['Fail','Pass']))

## 13. Model Comparison

In [ ]:
print("=== REGRESSION ==="); print(pd.DataFrame(reg_results).T.sort_values('R²',ascending=False))
print("\n=== CLASSIFICATION ==="); print(pd.DataFrame(clf_results).T.sort_values('ROC-AUC',ascending=False))
colors=['steelblue','darkorange','green']
fig,axes=plt.subplots(1,4,figsize=(18,4))
reg_df=pd.DataFrame(reg_results).T
clf_df=pd.DataFrame(clf_results).T
axes[0].barh(reg_df.index,reg_df['RMSE'],color='#e05c5c',edgecolor='white'); axes[0].set_title('Reg — RMSE')
axes[1].barh(reg_df.index,reg_df['R²'],color='#5c9ee0',edgecolor='white'); axes[1].set_title('Reg — R²')
axes[2].barh(clf_df.index,clf_df['Accuracy'],color='#2ecc71',edgecolor='white'); axes[2].set_title('Clf — Accuracy')
for (name,ypr),color in zip(clf_probs.items(),colors):
    fpr,tpr,_=roc_curve(yc_test,ypr)
    axes[3].plot(fpr,tpr,label=f"{name} ({clf_results[name]['ROC-AUC']:.3f})",color=color,linewidth=2)
axes[3].plot([0,1],[0,1],'k--',linewidth=1); axes[3].set_title('Clf — ROC Curves'); axes[3].legend(fontsize=7)
plt.suptitle('Full Model Comparison',fontsize=13,y=1.02); plt.tight_layout(); plt.show()

---
## 🔮 14. Predict Score & Pass/Fail for Your Own Student
**Edit the values below and run the cell.**

In [ ]:
# ╔══════════════════════════════════════════╗
# ║   ✏️  CHANGE THESE VALUES TO YOUR INPUT  ║
# ╚══════════════════════════════════════════╝

gender                    = 'female'           # 'male' or 'female'
race_ethnicity            = 'group C'          # 'group A','group B','group C','group D','group E'
parental_level_of_education = "bachelor's degree"
                                               # 'some high school','high school','some college',
                                               # "associate's degree","bachelor's degree","master's degree"
lunch                     = 'standard'         # 'standard' or 'free/reduced'
test_preparation_course   = 'completed'        # 'completed' or 'none'

# ── Auto-process ─────────────────────────
new_row = pd.DataFrame([{
    'gender'                      : gender,
    'race/ethnicity'              : race_ethnicity,
    'parental level of education' : parental_level_of_education,
    'lunch'                       : lunch,
    'test preparation course'     : test_preparation_course
}])

new_enc = pd.get_dummies(new_row)
new_enc = new_enc.reindex(columns=FEATURE_COLS, fill_value=0)
new_scaled = scaler.transform(new_enc)

print("=" * 50)
print("       🎓 STUDENT PERFORMANCE PREDICTION")
print("=" * 50)
print(f"  Gender              : {gender}")
print(f"  Race/Ethnicity      : {race_ethnicity}")
print(f"  Parental Education  : {parental_level_of_education}")
print(f"  Lunch               : {lunch}")
print(f"  Test Prep           : {test_preparation_course}")
print("-" * 50)
print("  📈 REGRESSION — Predicted Average Score:")
for name, model in reg_models.items():
    pred = model.predict(new_scaled)[0]
    print(f"    {name:<22}: {pred:.1f} / 100")
print("-" * 50)
print("  ✅ CLASSIFICATION — Pass / Fail:")
for name, model in clf_models.items():
    pred = model.predict(new_scaled)[0]
    prob = model.predict_proba(new_scaled)[0][1]
    label = '✅ PASS' if pred == 1 else '❌ FAIL'
    print(f"    {name:<22}: {label}  (pass prob: {prob:.2%})")
print("=" * 50)